# OpenWeedLocator ONNX Model Setup and Training

This notebook provides a comprehensive guide for setting up, training, and deploying ONNX models for weed detection using OpenWeedLocator. This replaces the previous Google Coral workflow with a pure ONNX approach that works on Raspberry Pi 5 and other systems.

## Overview

- **Install ONNX Runtime and dependencies**
- **Load and test ONNX models**
- **Configure model paths and labels**
- **Train custom YOLOv8 models**
- **Export models to ONNX format**
- **Set up data collection**
- **Test model performance**

## 1. Install ONNX Runtime and Dependencies

First, we'll install ONNX Runtime and other required libraries for model training and inference.

In [ ]:
# Install ONNX Runtime
!pip install onnxruntime

# Install Ultralytics for YOLOv8 training and export
!pip install ultralytics

# Install additional dependencies
!pip install opencv-python numpy pillow matplotlib

print("✅ Installation complete!")

In [ ]:
# Verify installations
import onnxruntime as ort
import ultralytics
import cv2
import numpy as np
from pathlib import Path

print(f"ONNX Runtime version: {ort.__version__}")
print(f"Ultralytics version: {ultralytics.__version__}")
print(f"OpenCV version: {cv2.__version__}")
print(f"Available providers: {ort.get_available_providers()}")

## 2. Load and Test ONNX Model

Test loading an ONNX model and inspect its properties to ensure proper setup.

In [ ]:
def test_onnx_model(model_path):
    """Test loading an ONNX model and display its properties."""
    try:
        # Create inference session
        session = ort.InferenceSession(model_path, providers=['CPUExecutionProvider'])
        
        # Get model inputs and outputs
        inputs = session.get_inputs()
        outputs = session.get_outputs()
        
        print(f"✅ Successfully loaded model: {model_path}")
        print(f"\nModel Inputs:")
        for inp in inputs:
            print(f"  - Name: {inp.name}, Type: {inp.type}, Shape: {inp.shape}")
        
        print(f"\nModel Outputs:")
        for out in outputs:
            print(f"  - Name: {out.name}, Type: {out.type}, Shape: {out.shape}")
        
        return session
    
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        return None

# Test with your model (update path as needed)
model_path = "weed_model-v1 5nu.onnx"  # Update this path

if Path(model_path).exists():
    session = test_onnx_model(model_path)
else:
    print(f"❌ Model file not found: {model_path}")
    print("Available files in current directory:")
    for file in Path(".").glob("*.onnx"):
        print(f"  - {file.name}")

In [ ]:
# Test inference with dummy data
if 'session' in locals() and session is not None:
    try:
        # Get input shape from model
        input_name = session.get_inputs()[0].name
        input_shape = session.get_inputs()[0].shape
        
        # Handle dynamic batch size
        if input_shape[0] == 'batch' or input_shape[0] is None:
            input_shape[0] = 1
        
        print(f"Creating dummy input with shape: {input_shape}")
        
        # Create dummy input data
        dummy_input = np.random.rand(*input_shape).astype(np.float32)
        
        # Run inference
        outputs = session.run(None, {input_name: dummy_input})
        
        print(f"✅ Inference successful!")
        print(f"Output shapes: {[out.shape for out in outputs]}")
        
    except Exception as e:
        print(f"❌ Inference failed: {e}")

## 3. Configure Model Paths and Labels

Set up the model directory structure and configure labels for the OpenWeedLocator system.

In [ ]:
# Create proper directory structure
models_dir = Path("../models")  # Relative to notebook location
models_dir.mkdir(exist_ok=True)

print(f"Models directory: {models_dir.absolute()}")
print(f"Directory exists: {models_dir.exists()}")

# List existing ONNX models
onnx_models = list(models_dir.glob("*.onnx"))
print(f"\nFound {len(onnx_models)} ONNX models:")
for model in onnx_models:
    print(f"  - {model.name}")

In [ ]:
# Create or update labels.txt file
labels_file = models_dir / "labels.txt"

# Example labels for weed detection (customize as needed)
default_labels = [
    "weed",
    "crop",
    "background"
]

# Create labels file
with open(labels_file, 'w') as f:
    for label in default_labels:
        f.write(f"{label}\n")

print(f"✅ Created labels file: {labels_file}")
print(f"Labels: {default_labels}")

# Display file contents
if labels_file.exists():
    with open(labels_file, 'r') as f:
        content = f.read()
    print(f"\nLabels file content:\n{content}")

In [ ]:
# Create ONNX config file for OWL
config_dir = Path("../config")
config_dir.mkdir(exist_ok=True)

onnx_config_content = """
[System]
algorithm = gog
relay_num = 4
actuation_duration = 0.15
delay = 0

[Camera]
resolution_width = 640
resolution_height = 480
exp_compensation = -2

[GreenOnGreen]
model_path = models
confidence = 0.5
class_filter_id = 0

[DataCollection]
sample_images = False
sample_method = whole
sample_frequency = 30
save_directory = /media/owl/SanDisk
disable_detection = False
log_fps = False
camera_name = cam1

[Relays]
0 = 13
1 = 15
2 = 16
3 = 18
""".strip()

config_file = config_dir / "ONNX_GOG.ini"
with open(config_file, 'w') as f:
    f.write(onnx_config_content)

print(f"✅ Created ONNX config file: {config_file}")

## 4. Train YOLOv8 Model for Weed Detection

Train a custom YOLOv8 model using your dataset with proper configuration for weed and crop classification.

In [ ]:
# Example dataset configuration
# You'll need to replace this with your actual dataset

dataset_config = """
# YOLOv8 dataset configuration
path: /path/to/your/dataset  # Update this path
train: images/train
val: images/val
test: images/test  # optional

# Classes
nc: 3  # number of classes
names: ['weed', 'crop', 'background']
"""

# Save dataset config
dataset_file = Path("weed_dataset.yaml")
with open(dataset_file, 'w') as f:
    f.write(dataset_config.strip())

print(f"✅ Created dataset config: {dataset_file}")
print("\n⚠️  Remember to update the dataset path in weed_dataset.yaml")
print("   Point it to your actual annotated weed detection dataset")

In [ ]:
# Training example (uncomment when you have a dataset)
from ultralytics import YOLO

print("Example YOLOv8 training command:")
print("")
print("# Load a YOLOv8 model")
print("model = YOLO('yolov8n.pt')  # nano model for faster training")
print("")
print("# Train the model")
print("results = model.train(")
print("    data='weed_dataset.yaml',  # path to dataset config")
print("    epochs=100,               # number of training epochs")
print("    imgsz=640,               # image size")
print("    device='cpu',            # use CPU (or 'cuda' if available)")
print("    patience=10,             # early stopping patience")
print("    save=True,               # save checkpoints")
print("    cache=True               # cache images for faster training")
print(")")
print("")
print("⚠️  Training requires an annotated dataset in YOLO format")
print("   Use tools like Roboflow or CVAT to create annotations")

## 5. Export Trained Model to ONNX Format

Convert the trained YOLOv8 model to ONNX format for use with OpenWeedLocator.

In [ ]:
# Example model export (update path to your trained model)
from ultralytics import YOLO

def export_model_to_onnx(model_path, output_name="weed_model.onnx"):
    """Export a trained YOLOv8 model to ONNX format."""
    try:
        print(f"Loading model: {model_path}")
        model = YOLO(model_path)
        
        print("Exporting to ONNX format...")
        # Export to ONNX
        model.export(
            format='onnx',
            imgsz=640,
            simplify=True,  # Simplify the model for better compatibility
            opset=11        # ONNX opset version
        )
        
        # Find the exported ONNX file
        model_dir = Path(model_path).parent
        onnx_files = list(model_dir.glob("*.onnx"))
        
        if onnx_files:
            exported_file = onnx_files[-1]  # Get the most recent one
            target_file = models_dir / output_name
            
            # Copy to models directory
            import shutil
            shutil.copy2(exported_file, target_file)
            
            print(f"✅ Model exported successfully to: {target_file}")
            return target_file
        else:
            print("❌ No ONNX file found after export")
            return None
            
    except Exception as e:
        print(f"❌ Export failed: {e}")
        return None

# Example usage (uncomment when you have a trained model)
print("Example export command:")
print("export_model_to_onnx('runs/detect/train/weights/best.pt')")
print("")
print("This will:")
print("1. Load your trained YOLOv8 model")
print("2. Export it to ONNX format")
print("3. Copy it to the models directory")
print("4. Make it ready for OpenWeedLocator")

In [ ]:
# Verify exported model
def verify_exported_model(model_path):
    """Verify the exported ONNX model works correctly."""
    if not Path(model_path).exists():
        print(f"❌ Model file not found: {model_path}")
        return False
    
    try:
        # Load and test the model
        session = ort.InferenceSession(model_path, providers=['CPUExecutionProvider'])
        
        # Get input details
        input_name = session.get_inputs()[0].name
        input_shape = session.get_inputs()[0].shape
        
        # Create test input
        if input_shape[0] == 'batch' or input_shape[0] is None:
            input_shape[0] = 1
        
        test_input = np.random.rand(*input_shape).astype(np.float32)
        
        # Run inference
        outputs = session.run(None, {input_name: test_input})
        
        print(f"✅ Model verification successful!")
        print(f"Input shape: {input_shape}")
        print(f"Output shapes: {[out.shape for out in outputs]}")
        print(f"Model ready for OpenWeedLocator deployment!")
        
        return True
        
    except Exception as e:
        print(f"❌ Model verification failed: {e}")
        return False

# Check if we have any ONNX models to verify
onnx_models = list(models_dir.glob("*.onnx"))
if onnx_models:
    print(f"Verifying model: {onnx_models[0]}")
    verify_exported_model(onnx_models[0])
else:
    print("No ONNX models found to verify")

## 6. Implement Data Collection for Training

Set up automated data collection using OWL's sampling functionality to build training datasets.

In [ ]:
# Create data collection configuration
data_collection_config = """
[System]
algorithm = exhsv
relay_num = 4
actuation_duration = 0.15
delay = 0

[Camera]
resolution_width = 640
resolution_height = 480
exp_compensation = -2

[DataCollection]
# Enable data collection
sample_images = True
sample_method = whole
sample_frequency = 30
save_directory = /media/owl/SanDisk
disable_detection = True  # Disable detection for pure data collection
log_fps = True
camera_name = cam1

[GreenOnBrown]
exg_min = 25
exg_max = 200
hue_min = 39
hue_max = 83
saturation_min = 50
saturation_max = 220
brightness_min = 60
brightness_max = 190
min_detection_area = 10
invert_hue = False

[Relays]
0 = 13
1 = 15
2 = 16
3 = 18
""".strip()

# Save data collection config
data_config_file = config_dir / "DATA_COLLECTION.ini"
with open(data_config_file, 'w') as f:
    f.write(data_collection_config)

print(f"✅ Created data collection config: {data_config_file}")
print("")
print("To use for data collection:")
print("1. Insert USB drive in Raspberry Pi")
print("2. Update owl.py to use DATA_COLLECTION.ini")
print("3. Run OWL in the field to collect images")
print("4. Images will be saved every 30 frames to USB drive")

In [ ]:
# Data collection workflow guide
workflow_guide = """
# Data Collection Workflow for Training

## Step 1: Collect Raw Images
- Use DATA_COLLECTION.ini config
- Drive OWL through fields with weeds and crops
- Collect diverse lighting/weather conditions
- Aim for 1000+ images minimum

## Step 2: Organize Images
- Create train/val/test splits (70/20/10)
- Organize in YOLOv8 format:
  dataset/
  ├── images/
  │   ├── train/
  │   ├── val/
  │   └── test/
  └── labels/
      ├── train/
      ├── val/
      └── test/

## Step 3: Annotate Images
- Use Roboflow, CVAT, or LabelImg
- Draw bounding boxes around:
  - Weeds (class 0)
  - Crops (class 1)
  - Other objects (class 2)
- Export annotations in YOLO format

## Step 4: Train Model
- Use YOLOv8 training code above
- Monitor training metrics
- Adjust hyperparameters as needed

## Step 5: Export and Deploy
- Export best model to ONNX
- Copy to models directory
- Test with real field conditions
""".strip()

# Save workflow guide
workflow_file = Path("DATA_COLLECTION_WORKFLOW.md")
with open(workflow_file, 'w') as f:
    f.write(workflow_guide)

print(f"✅ Created workflow guide: {workflow_file}")
print("\nRefer to this file for detailed data collection steps")

## 7. Test Model Performance

Run inference tests with the ONNX model using the OpenWeedLocator system and evaluate detection performance.

In [ ]:
# Performance testing function
import time

def benchmark_model(model_path, num_iterations=100):
    """Benchmark ONNX model inference speed."""
    try:
        session = ort.InferenceSession(model_path, providers=['CPUExecutionProvider'])
        
        # Get input details
        input_name = session.get_inputs()[0].name
        input_shape = session.get_inputs()[0].shape
        
        if input_shape[0] == 'batch' or input_shape[0] is None:
            input_shape[0] = 1
        
        # Create test input
        test_input = np.random.rand(*input_shape).astype(np.float32)
        
        # Warm up
        for _ in range(10):
            session.run(None, {input_name: test_input})
        
        # Benchmark
        start_time = time.time()
        for _ in range(num_iterations):
            outputs = session.run(None, {input_name: test_input})
        end_time = time.time()
        
        total_time = end_time - start_time
        avg_time = total_time / num_iterations
        fps = 1.0 / avg_time
        
        print(f"✅ Benchmark Results:")
        print(f"Model: {Path(model_path).name}")
        print(f"Input shape: {input_shape}")
        print(f"Iterations: {num_iterations}")
        print(f"Total time: {total_time:.3f}s")
        print(f"Average time per inference: {avg_time*1000:.1f}ms")
        print(f"Estimated FPS: {fps:.1f}")
        
        # Performance assessment
        if fps >= 10:
            print(f"🚀 Excellent performance for real-time detection!")
        elif fps >= 5:
            print(f"✅ Good performance for field use")
        else:
            print(f"⚠️  Performance may be slow for real-time use")
        
        return fps
        
    except Exception as e:
        print(f"❌ Benchmark failed: {e}")
        return None

# Run benchmark on available models
onnx_models = list(models_dir.glob("*.onnx"))
if onnx_models:
    for model in onnx_models:
        print(f"\nBenchmarking: {model.name}")
        print("-" * 50)
        benchmark_model(model)
else:
    print("No ONNX models found for benchmarking")
    print("Add an ONNX model to the models directory to test performance")

In [ ]:
# Test with sample image (if available)
def test_model_with_image(model_path, image_path=None):
    """Test model with a real image."""
    try:
        session = ort.InferenceSession(model_path, providers=['CPUExecutionProvider'])
        
        # Get input details
        input_name = session.get_inputs()[0].name
        input_shape = session.get_inputs()[0].shape
        
        if input_shape[0] == 'batch' or input_shape[0] is None:
            input_shape[0] = 1
        
        if image_path and Path(image_path).exists():
            # Load and preprocess real image
            img = cv2.imread(str(image_path))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # Resize to model input size
            target_size = (input_shape[3], input_shape[2])  # W, H
            img_resized = cv2.resize(img, target_size)
            
            # Normalize and reshape
            img_normalized = img_resized.astype(np.float32) / 255.0
            img_input = np.transpose(img_normalized, (2, 0, 1))  # HWC to CHW
            img_input = np.expand_dims(img_input, axis=0)  # Add batch dimension
            
            print(f"Using real image: {image_path}")
        else:
            # Create synthetic test image
            img_input = np.random.rand(*input_shape).astype(np.float32)
            print("Using synthetic test image")
        
        # Run inference
        outputs = session.run(None, {input_name: img_input})
        
        print(f"✅ Inference successful!")
        print(f"Input shape: {img_input.shape}")
        print(f"Number of outputs: {len(outputs)}")
        
        for i, output in enumerate(outputs):
            print(f"Output {i} shape: {output.shape}")
            # For YOLO models, output is typically detections
            if len(output.shape) == 3 and output.shape[2] > 5:
                # Looks like YOLO detection output
                detections = output[0]  # Remove batch dimension
                # Count detections above confidence threshold
                confidence_threshold = 0.5
                if detections.shape[1] > 4:  # Has confidence scores
                    confidences = detections[:, 4]
                    high_conf_detections = np.sum(confidences > confidence_threshold)
                    print(f"Detections above {confidence_threshold}: {high_conf_detections}")
        
        return outputs
        
    except Exception as e:
        print(f"❌ Image test failed: {e}")
        return None

# Test with first available model
if onnx_models:
    print(f"Testing model with sample data: {onnx_models[0].name}")
    print("-" * 50)
    test_model_with_image(onnx_models[0])
else:
    print("No ONNX models found for testing")

In [ ]:
# Final deployment checklist
deployment_checklist = """
# 🚀 OpenWeedLocator ONNX Deployment Checklist

## ✅ Pre-Deployment
- [ ] ONNX Runtime installed and tested
- [ ] Model file placed in models/ directory
- [ ] labels.txt file created with correct classes
- [ ] ONNX_GOG.ini config file configured
- [ ] Model performance benchmarked (>5 FPS recommended)
- [ ] Test inference successful with dummy data

## 🔧 Raspberry Pi Setup
- [ ] Copy models/ directory to Pi
- [ ] Copy config/ directory to Pi
- [ ] Install onnxruntime on Pi: pip install onnxruntime
- [ ] Update owl.py to use ONNX_GOG.ini
- [ ] Test camera functionality
- [ ] Test GPIO/relay connections

## 🌱 Field Testing
- [ ] Test detection accuracy in field conditions
- [ ] Verify appropriate detection sensitivity
- [ ] Check false positive/negative rates
- [ ] Adjust confidence thresholds if needed
- [ ] Test at different travel speeds

## 📊 Performance Monitoring
- [ ] Monitor inference speed (FPS)
- [ ] Check CPU/memory usage
- [ ] Validate detection accuracy
- [ ] Log any errors or issues

Your OpenWeedLocator ONNX system is ready for deployment! 🎉
""".strip()

print(deployment_checklist)

# Save checklist
checklist_file = Path("DEPLOYMENT_CHECKLIST.md")
with open(checklist_file, 'w') as f:
    f.write(deployment_checklist)

print(f"\n✅ Deployment checklist saved to: {checklist_file}")